# Merged BERTopic Step By Step

This notebook is intentionally incremental.

We will not run the full merged workflow in one go. Instead, we will:
1. load the saved outlet models from `1a_BERTopic/outputs/`
2. confirm the current per-outlet stats
3. merge **Tagesschau as the base** and add outlets one by one
4. inspect topic counts / labels after each step
5. decide together whether to reduce outliers, rename topics, or keep going
6. only later build the outlet-topic frequency table and outlet-colored UMAP

Run one section at a time and stop after each checkpoint.

## Step 0. Setup

This cell only sets paths, imports helpers, and loads the current saved outlet models.

Stop after running the next two code cells and inspect the printed overview before merging anything.

In [3]:
import os
import sys
import importlib
from pathlib import Path

import pandas as pd
from IPython.display import display
from bertopic import BERTopic

EMBEDDING_MODEL = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
MIN_SIMILARITY = 0.7


def find_project_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / ".git").exists():
            return candidate
    raise FileNotFoundError("Could not find project root containing .git")


PROJECT_ROOT = find_project_root(Path.cwd())
MODULE_ROOT = PROJECT_ROOT / "1a_BERTopic"
if str(MODULE_ROOT) not in sys.path:
    sys.path.insert(0, str(MODULE_ROOT))

os.environ["NUMBA_CACHE_DIR"] = str(PROJECT_ROOT / ".numba_cache")

import merged_outlets_analysis as moa
moa = importlib.reload(moa)

MODEL_DIR_CANDIDATES = [PROJECT_ROOT / "1a_BERTopic" / "outputs"]
OUTLET_SPECS = moa.OUTLET_SPECS
resolve_model_paths = moa.resolve_model_paths

MODEL_PATHS = resolve_model_paths(MODEL_DIR_CANDIDATES)
loaded_models = {
    key: BERTopic.load(model_path, embedding_model=EMBEDDING_MODEL)
    for key, model_path in MODEL_PATHS.items()
}

MERGE_ORDER = [
    "tagesschau",
    "rt",
    "antispiegel",
    "tichys",
    "nius",
    "compact",
    "deutschlandkurier",
]

print(f"Project root: {PROJECT_ROOT}")
print("Loaded models from:")
for key in MERGE_ORDER:
    print(f"- {key}: {MODEL_PATHS[key]}")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7791.08it/s]
BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 15312.45it/s]
BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 17355.62it/s]
BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | S

Project root: /Users/MattisHaumann/Dev/Thesis
Loaded models from:
- tagesschau: /Users/MattisHaumann/Dev/Thesis/1a_BERTopic/outputs/ts_model
- rt: /Users/MattisHaumann/Dev/Thesis/1a_BERTopic/outputs/rt_model
- antispiegel: /Users/MattisHaumann/Dev/Thesis/1a_BERTopic/outputs/as_model
- tichys: /Users/MattisHaumann/Dev/Thesis/1a_BERTopic/outputs/te_model
- nius: /Users/MattisHaumann/Dev/Thesis/1a_BERTopic/outputs/ns_model
- compact: /Users/MattisHaumann/Dev/Thesis/1a_BERTopic/outputs/compact_model
- deutschlandkurier: /Users/MattisHaumann/Dev/Thesis/1a_BERTopic/outputs/dk_model


In [4]:
raw_counts = pd.read_csv(PROJECT_ROOT / "00_Initial EDA" / "df_combined.csv")["source"].value_counts().to_dict()
source_name_map = {
    "tagesschau": "Tagesschau",
    "rt": "RT_de",
    "antispiegel": "Antispiegel",
    "tichys": "Tichys_Einblick",
    "nius": "Nius",
    "compact": "Compact",
    "deutschlandkurier": "Deutschlandkurier",
}

rows = []
for key in MERGE_ORDER:
    model = loaded_models[key]
    topic_info = model.get_topic_info().copy()
    docs = len(model.topics_)
    outliers = int((pd.Series(model.topics_) == -1).sum())
    topics = int(topic_info.loc[topic_info["Topic"] != -1, "Topic"].nunique())
    assigned = docs - outliers
    rows.append(
        {
            "Outlet": OUTLET_SPECS[key].label,
            "RawDocs": raw_counts[source_name_map[key]],
            "ModelDocs": docs,
            "Topics": topics,
            "OutlierPct": round(outliers / docs * 100, 2),
            "DocsPerTopic": round(assigned / topics, 2) if topics else float("nan"),
        }
    )

step0_summary = pd.DataFrame(rows)
display(step0_summary)

print("Base outlet for the cumulative merge:", OUTLET_SPECS[MERGE_ORDER[0]].label)
print("Next step after review: merge Tagesschau + RT and inspect the merged topic overview.")

,Outlet,RawDocs,ModelDocs,Topics,OutlierPct,DocsPerTopic
0,Tagesschau,6320,6319,55,11.77,101.36
1,RT,4560,4560,50,16.51,76.14
2,Antispiegel,565,565,17,1.42,32.76
3,Tichys Einblick,2756,2756,51,1.12,53.43
4,Nius,3269,3266,38,15.09,72.97
5,Compact,1486,1486,31,13.46,41.48
6,Deutschlandkurier,1484,1465,26,13.52,48.73


Base outlet for the cumulative merge: Tagesschau
Next step after review: merge Tagesschau + RT and inspect the merged topic overview.


## Step 1. Merge Tagesschau + RT

This is the first cumulative merge step.

Goal:
- use **Tagesschau as the base**
- add **RT**
- inspect the classic merged topic count and top topic overview before doing anything else

In [5]:
step1_keys = ["tagesschau", "rt"]
step1_labels = [OUTLET_SPECS[key].label for key in step1_keys]
step1_models = [loaded_models[key] for key in step1_keys]

step1_merged_model = BERTopic.merge_models(step1_models, min_similarity=MIN_SIMILARITY)
step1_topic_info = step1_merged_model.get_topic_info().copy()

step1_substantive = (
    step1_topic_info.loc[step1_topic_info["Topic"] != -1, ["Topic", "Count", "Name", "Representation"]]
    .sort_values(["Count", "Topic"], ascending=[False, True])
    .reset_index(drop=True)
)
step1_substantive.insert(0, "Rank", range(1, len(step1_substantive) + 1))

step1_summary = pd.DataFrame(
    [
        {
            "IncludedOutlets": " + ".join(step1_labels),
            "BaseOutlet": OUTLET_SPECS[step1_keys[0]].label,
            "AddedOutlet": OUTLET_SPECS[step1_keys[1]].label,
            "TagesschauTopics": int(step0_summary.loc[step0_summary["Outlet"] == OUTLET_SPECS["tagesschau"].label, "Topics"].iloc[0]),
            "RTTopics": int(step0_summary.loc[step0_summary["Outlet"] == OUTLET_SPECS["rt"].label, "Topics"].iloc[0]),
            "MergedTopics": int(step1_substantive["Topic"].nunique()),
            "MergedRowsIncludingOutlier": int(len(step1_topic_info)),
        }
    ]
)

display(step1_summary)
display(step1_substantive.head(30))

print("Merged topic count excluding -1:", int(step1_substantive['Topic'].nunique()))
print("Next decision after inspection: continue as-is, change min_similarity, or stop and reconsider outlet models.")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 14443.10it/s]
BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


,IncludedOutlets,BaseOutlet,AddedOutlet,TagesschauTopics,RTTopics,MergedTopics,MergedRowsIncludingOutlier
0,Tagesschau + RT,Tagesschau,RT,55,50,57,58


,Rank,Topic,Count,Name,Representation
0,1,0,784,0_israel_hamas_gazastreifen_gaza,"[israel, hamas, gazastreifen, gaza, geiseln, i..."
1,2,1,716,1_ukraine_putin_selenskyj_russland,"[ukraine, putin, selenskyj, russland, trump, r..."
2,3,48,601,48_ukraine_russland_russische_russischen,"[ukraine, russland, russische, russischen, ukr..."
3,4,3,438,3_venezuela_maduro_trump_venezuelas,"[venezuela, maduro, trump, venezuelas, machado..."
4,5,24,411,24_russland_ukraine_russischen_russische,"[russland, ukraine, russischen, russische, san..."
5,6,15,367,15_bsw_afd_cdu_spd,"[bsw, afd, cdu, spd, grünen, wahl, koalition, ..."
6,7,7,329,7_china_xi_indien_peking,"[china, xi, indien, peking, trump, chinas, chi..."
7,8,47,309,47_nato_drohnen_polen_luftraum,"[nato, drohnen, polen, luftraum, polnischen, r..."
8,9,2,302,2_dollar_dax_fed_anleger,"[dollar, dax, fed, anleger, punkten, aktien, a..."
9,10,5,234,5_bundesanwaltschaft_gericht_haft_prozess,"[bundesanwaltschaft, gericht, haft, prozess, s..."


Merged topic count excluding -1: 57
Next decision after inspection: continue as-is, change min_similarity, or stop and reconsider outlet models.


In [7]:
SUSPECT_TERMS = ("russ", "ukrain", "kiew", "selensky", "putin", "moskau")

def topic_words_string(model, topic_id, top_n=30):
    words = model.get_topic(int(topic_id)) or []
    return ", ".join(word for word, _ in words[:top_n])

def get_representative_docs_safe(model, topic_id, n=3):
    try:
        docs = model.get_representative_docs(int(topic_id))
        if isinstance(docs, list):
            return docs[:n]
    except TypeError:
        pass
    except Exception:
        pass

    try:
        docs_map = model.get_representative_docs()
        docs = docs_map.get(int(topic_id), [])
        return docs[:n] if isinstance(docs, list) else []
    except Exception:
        return []

suspect_topics = step1_substantive.loc[
    step1_substantive.apply(
        lambda row: any(
            term in f"{str(row['Name']).lower()} {topic_words_string(step1_merged_model, row['Topic']).lower()}"
            for term in SUSPECT_TERMS
        ),
        axis=1,
    ),
    ["Rank", "Topic", "Count", "Name"],
].copy()

suspect_topics["TopWords"] = suspect_topics["Topic"].apply(
    lambda topic_id: topic_words_string(step1_merged_model, topic_id, top_n=12)
)

display(suspect_topics)

TOPIC_IDS_TO_INSPECT = suspect_topics["Topic"].head(4).tolist()

for topic_id in TOPIC_IDS_TO_INSPECT:
    row = suspect_topics.loc[suspect_topics["Topic"] == topic_id].iloc[0]
    print("=" * 120)
    print(f"Topic {int(topic_id)} | count={int(row['Count'])} | {row['Name']}")
    print("Top words:", row["TopWords"])

    rep_docs = get_representative_docs_safe(step1_merged_model, topic_id, n=3)
    if not rep_docs:
        print("Representative docs not available from the saved model.")
    else:
        for i, doc in enumerate(rep_docs, 1):
            print(f"[Doc {i}] {str(doc)[:700].replace('\\n', ' ')}")
    print()


,Rank,Topic,Count,Name,TopWords
1,2,1,716,1_ukraine_putin_selenskyj_russland,"ukraine, putin, selenskyj, russland, trump, ru..."
2,3,48,601,48_ukraine_russland_russische_russischen,"ukraine, russland, russische, russischen, ukra..."
4,5,24,411,24_russland_ukraine_russischen_russische,"russland, ukraine, russischen, russische, sank..."
6,7,7,329,7_china_xi_indien_peking,"china, xi, indien, peking, trump, chinas, chin..."
7,8,47,309,47_nato_drohnen_polen_luftraum,"nato, drohnen, polen, luftraum, polnischen, ru..."
16,17,28,168,28_merz_kanzler_stadtbild_mercosur,"merz, kanzler, stadtbild, mercosur, bundeskanz..."
23,24,18,111,18_ukraine_russland_russischen_kiew,"ukraine, russland, russischen, kiew, ukrainisc..."
38,39,55,77,55_indische_modi_delhi_neu delhi,"indische, modi, delhi, neu delhi, indiens, öl,..."


Topic 1 | count=716 | 1_ukraine_putin_selenskyj_russland
Top words: ukraine, putin, selenskyj, russland, trump, russischen, ukrainischen, ukrainische, sicherheitsgarantien, us präsident, russische, präsidenten
Representative docs not available from the saved model.

Topic 48 | count=601 | 48_ukraine_russland_russische_russischen
Top words: ukraine, russland, russische, russischen, ukrainische, ukrainischen, pokrowsk, soldaten, truppen, front, putin, region
Representative docs not available from the saved model.

Topic 24 | count=411 | 24_russland_ukraine_russischen_russische
Top words: russland, ukraine, russischen, russische, sanktionen, merz, belgien, euroclear, eu staaten, gas, vermögen, russisches
Representative docs not available from the saved model.

Topic 7 | count=329 | 7_china_xi_indien_peking
Top words: china, xi, indien, peking, trump, chinas, chinesische, taiwan, g20, südafrika, japan, chinesischen
Representative docs not available from the saved model.



In [8]:
pd.set_option("display.max_colwidth", None)

suspect_topics_30 = suspect_topics.copy()
suspect_topics_30["Top30Words"] = suspect_topics_30["Topic"].apply(
    lambda topic_id: topic_words_string(step1_merged_model, topic_id, top_n=30)
)

display(suspect_topics_30[["Rank", "Topic", "Count", "Name", "Top30Words"]])


,Rank,Topic,Count,Name,Top30Words
1,2,1,716,1_ukraine_putin_selenskyj_russland,"ukraine, putin, selenskyj, russland, trump, russischen, ukrainischen, ukrainische, sicherheitsgarantien, us präsident, russische, präsidenten, nato, moskau, wladimir, wladimir putin, wolodymyr, wolodymyr selenskyj, krieg, donald, donald trump, alaska, frieden, für ukraine, merz, russlands, präsident donald, gespräche, europäer, kiew"
2,3,48,601,48_ukraine_russland_russische_russischen,"ukraine, russland, russische, russischen, ukrainische, ukrainischen, pokrowsk, soldaten, truppen, front, putin, region, selenskyj, krieg, kiew, drohnen, donezk, russischer, armee, belarus, moskau, manöver, russlands, donbass, ukrainer, sapad, telegram, streitkräfte, wolodymyr selenskyj, mariupol"
4,5,24,411,24_russland_ukraine_russischen_russische,"russland, ukraine, russischen, russische, sanktionen, merz, belgien, euroclear, eu staaten, gas, vermögen, russisches, brüssel, ungarn, kommission, milliarden euro, nutzung, gipfel, für ukraine, russlands, vermögenswerte, leyen, putin, slowakei, wever, de wever, orban, eingefrorenen, lng, bundeskanzler"
6,7,7,329,7_china_xi_indien_peking,"china, xi, indien, peking, trump, chinas, chinesische, taiwan, g20, südafrika, japan, chinesischen, tiktok, erden, us präsident, zölle, gipfel, südkorea, beziehungen, xi jinping, jinping, russland, wadephul, modi, tests, putin, nordkorea, volksrepublik, donald, trumps"
7,8,47,309,47_nato_drohnen_polen_luftraum,"nato, drohnen, polen, luftraum, polnischen, russland, russische, ukraine, russischen, polens, polnischen luftraum, luftraums, tusk, russische drohnen, vorfall, eindringen, russlands, polnische, belarus, ostflanke, warschau, russischer, kampfjets, drohne, eingedrungen, drohnen polnischen, verletzung, litauen, russischer drohnen, pistorius"
16,17,28,168,28_merz_kanzler_stadtbild_mercosur,"merz, kanzler, stadtbild, mercosur, bundeskanzler, afd, friedrich, friedrich merz, abkommen, cdu, grünen, spd, merkel, migration, weber, kommission, kanzler merz, bundesregierung, parlament, sagte merz, leyen, regierungserklärung, bundestag, aussagen, ukraine, bundeskanzler friedrich, kanzlers, französischen, töchter, macron"
23,24,18,111,18_ukraine_russland_russischen_kiew,"ukraine, russland, russischen, kiew, ukrainische, selenskyj, ukrainischen, russische, drohnen, angriffe, angriff, region, raketen, moskau, angriffen, getötet, telegram, nacht, wolodymyr, strom, beschädigt, wolodymyr selenskyj, sanktionen, verletzt, odessa, putin, russlands, präsident wolodymyr, angegriffen, hauptstadt"
38,39,55,77,55_indische_modi_delhi_neu delhi,"indische, modi, delhi, neu delhi, indiens, öl, indischen, russischem, russischem öl, zölle, narendra, narendra modi, soz, russisches, premierminister narendra, russisches öl, india, russland indien, raffinerien, jaishankar, xi, öls, barrel, indien china, russischen öls, importe, gipfel, china indien, handel, kaufen"


In [9]:
step2_keys = ["tagesschau", "rt", "antispiegel"]
step2_labels = [OUTLET_SPECS[key].label for key in step2_keys]
step2_models = [loaded_models[key] for key in step2_keys]

step2_merged_model = BERTopic.merge_models(step2_models, min_similarity=MIN_SIMILARITY)
step2_topic_info = step2_merged_model.get_topic_info().copy()

step2_substantive = (
    step2_topic_info.loc[step2_topic_info["Topic"] != -1, ["Topic", "Count", "Name", "Representation"]]
    .sort_values(["Count", "Topic"], ascending=[False, True])
    .reset_index(drop=True)
)
step2_substantive.insert(0, "Rank", range(1, len(step2_substantive) + 1))

step2_summary = pd.DataFrame(
    [
        {
            "IncludedOutlets": " + ".join(step2_labels),
            "BaseOutlet": OUTLET_SPECS[step2_keys[0]].label,
            "NewlyAddedOutlet": OUTLET_SPECS[step2_keys[-1]].label,
            "PreviousMergedTopics": int(step1_substantive["Topic"].nunique()),
            "AddedOutletTopics": int(step0_summary.loc[step0_summary["Outlet"] == OUTLET_SPECS["antispiegel"].label, "Topics"].iloc[0]),
            "MergedTopicsNow": int(step2_substantive["Topic"].nunique()),
            "MergedRowsIncludingOutlier": int(len(step2_topic_info)),
        }
    ]
)

display(step2_summary)
display(step2_substantive.head(30))

print("Merged topic count excluding -1:", int(step2_substantive["Topic"].nunique()))
print("Decision after inspection: continue, adjust min_similarity, or stop and inspect topic overlaps more closely.")


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2494.61it/s]
BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


,IncludedOutlets,BaseOutlet,NewlyAddedOutlet,PreviousMergedTopics,AddedOutletTopics,MergedTopicsNow,MergedRowsIncludingOutlier
0,Tagesschau + RT + Antispiegel,Tagesschau,Antispiegel,57,17,58,59


,Rank,Topic,Count,Name,Representation
0,1,1,872,1_ukraine_putin_selenskyj_russland,"[ukraine, putin, selenskyj, russland, trump, russischen, ukrainischen, ukrainische, sicherheitsgarantien, us präsident, russische, präsidenten, nato, moskau, wladimir, wladimir putin, wolodymyr, wolodymyr selenskyj, krieg, donald, donald trump, alaska, frieden, für ukraine, merz, russlands, präsident donald, gespräche, europäer, kiew]"
1,2,0,809,0_israel_hamas_gazastreifen_gaza,"[israel, hamas, gazastreifen, gaza, geiseln, israelischen, israelische, israels, netanjahu, un, armee, waffenruhe, palästinensischen, al, freilassung, angriff, benjamin netanjahu, palästinenser, katar, krieg, benjamin, terrororganisation, getötet, trump, gazastreifens, ägypten, gaza stadt, palästinensische, plan, hisbollah]"
2,3,48,711,48_ukraine_russland_russische_russischen,"[ukraine, russland, russische, russischen, ukrainische, ukrainischen, pokrowsk, soldaten, truppen, front, putin, region, selenskyj, krieg, kiew, drohnen, donezk, russischer, armee, belarus, moskau, manöver, russlands, donbass, ukrainer, sapad, telegram, streitkräfte, wolodymyr selenskyj, mariupol]"
3,4,24,497,24_russland_ukraine_russischen_russische,"[russland, ukraine, russischen, russische, sanktionen, merz, belgien, euroclear, eu staaten, gas, vermögen, russisches, brüssel, ungarn, kommission, milliarden euro, nutzung, gipfel, für ukraine, russlands, vermögenswerte, leyen, putin, slowakei, wever, de wever, orban, eingefrorenen, lng, bundeskanzler]"
4,5,3,438,3_venezuela_maduro_trump_venezuelas,"[venezuela, maduro, trump, venezuelas, machado, bolsonaro, nicolás, nicolás maduro, karibik, venezolanische, venezolanischen, präsidenten, angriff, caracas, drogen, kolumbien, us präsident, donald trump, öl, donald, us regierung, präsident donald, milei, boote, vereinigten staaten, washington, staatschef, un, brasilien, moraes]"
5,6,15,367,15_bsw_afd_cdu_spd,"[bsw, afd, cdu, spd, grünen, wahl, koalition, bundestag, kandidaten, gersdorf, brosius, brosius gersdorf, wagenknecht, fraktion, kandidatin, schulze, stimmen, sachsen, bundesverfassungsgericht, linken, mehrheit, linke, haseloff, zweidrittelmehrheit, richter, sachsen anhalt, anhalt, ministerpräsident, gewählt, brandenburg]"
6,7,47,333,47_nato_drohnen_polen_luftraum,"[nato, drohnen, polen, luftraum, polnischen, russland, russische, ukraine, russischen, polens, polnischen luftraum, luftraums, tusk, russische drohnen, vorfall, eindringen, russlands, polnische, belarus, ostflanke, warschau, russischer, kampfjets, drohne, eingedrungen, drohnen polnischen, verletzung, litauen, russischer drohnen, pistorius]"
7,8,7,329,7_china_xi_indien_peking,"[china, xi, indien, peking, trump, chinas, chinesische, taiwan, g20, südafrika, japan, chinesischen, tiktok, erden, us präsident, zölle, gipfel, südkorea, beziehungen, xi jinping, jinping, russland, wadephul, modi, tests, putin, nordkorea, volksrepublik, donald, trumps]"
8,9,2,302,2_dollar_dax_fed_anleger,"[dollar, dax, fed, anleger, punkten, aktien, aktie, ki, notenbank, börse, leitindex, quartal, milliarden dollar, nvidia, wall street, wall, street, punkte, handel, konzern, us notenbank, nasdaq, powell, gold, index, marke, dow, zinsen, trump, börsen]"
9,10,5,234,5_bundesanwaltschaft_gericht_haft_prozess,"[bundesanwaltschaft, gericht, haft, prozess, sarkozy, mann, ermittler, tat, verurteilt, staatsanwaltschaft, urteil, stream, nord stream, ermittlungen, festgenommen, nord, anklage, angeklagten, festnahme, polizei, benko, täter, untersuchungshaft, marsalek, angeklagte, mutmaßlichen, landgericht, krah, anschlag, pipelines]"


Merged topic count excluding -1: 58
Decision after inspection: continue, adjust min_similarity, or stop and inspect topic overlaps more closely.


In [10]:
step3_keys = ["tagesschau", "rt", "antispiegel", "tichys"]
step3_labels = [OUTLET_SPECS[key].label for key in step3_keys]
step3_models = [loaded_models[key] for key in step3_keys]

step3_merged_model = BERTopic.merge_models(step3_models, min_similarity=MIN_SIMILARITY)
step3_topic_info = step3_merged_model.get_topic_info().copy()

step3_substantive = (
    step3_topic_info.loc[step3_topic_info["Topic"] != -1, ["Topic", "Count", "Name", "Representation"]]
    .sort_values(["Count", "Topic"], ascending=[False, True])
    .reset_index(drop=True)
)
step3_substantive.insert(0, "Rank", range(1, len(step3_substantive) + 1))

step3_summary = pd.DataFrame(
    [
        {
            "IncludedOutlets": " + ".join(step3_labels),
            "BaseOutlet": OUTLET_SPECS[step3_keys[0]].label,
            "NewlyAddedOutlet": OUTLET_SPECS[step3_keys[-1]].label,
            "PreviousMergedTopics": int(step2_substantive["Topic"].nunique()),
            "AddedOutletTopics": int(step0_summary.loc[step0_summary["Outlet"] == OUTLET_SPECS["tichys"].label, "Topics"].iloc[0]),
            "MergedTopicsNow": int(step3_substantive["Topic"].nunique()),
            "MergedRowsIncludingOutlier": int(len(step3_topic_info)),
        }
    ]
)

display(step3_summary)
display(step3_substantive.head(30))

print("Merged topic count excluding -1:", int(step3_substantive["Topic"].nunique()))
print("Decision after inspection: continue, adjust min_similarity, or inspect topic overlaps more closely.")


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6491.82it/s]
BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


,IncludedOutlets,BaseOutlet,NewlyAddedOutlet,PreviousMergedTopics,AddedOutletTopics,MergedTopicsNow,MergedRowsIncludingOutlier
0,Tagesschau + RT + Antispiegel + Tichys Einblick,Tagesschau,Tichys Einblick,58,51,64,65


,Rank,Topic,Count,Name,Representation
0,1,1,1023,1_ukraine_putin_selenskyj_russland,"[ukraine, putin, selenskyj, russland, trump, russischen, ukrainischen, ukrainische, sicherheitsgarantien, us präsident, russische, präsidenten, nato, moskau, wladimir, wladimir putin, wolodymyr, wolodymyr selenskyj, krieg, donald, donald trump, alaska, frieden, für ukraine, merz, russlands, präsident donald, gespräche, europäer, kiew]"
1,2,0,948,0_israel_hamas_gazastreifen_gaza,"[israel, hamas, gazastreifen, gaza, geiseln, israelischen, israelische, israels, netanjahu, un, armee, waffenruhe, palästinensischen, al, freilassung, angriff, benjamin netanjahu, palästinenser, katar, krieg, benjamin, terrororganisation, getötet, trump, gazastreifens, ägypten, gaza stadt, palästinensische, plan, hisbollah]"
2,3,48,711,48_ukraine_russland_russische_russischen,"[ukraine, russland, russische, russischen, ukrainische, ukrainischen, pokrowsk, soldaten, truppen, front, putin, region, selenskyj, krieg, kiew, drohnen, donezk, russischer, armee, belarus, moskau, manöver, russlands, donbass, ukrainer, sapad, telegram, streitkräfte, wolodymyr selenskyj, mariupol]"
3,4,15,693,15_bsw_afd_cdu_spd,"[bsw, afd, cdu, spd, grünen, wahl, koalition, bundestag, kandidaten, gersdorf, brosius, brosius gersdorf, wagenknecht, fraktion, kandidatin, schulze, stimmen, sachsen, bundesverfassungsgericht, linken, mehrheit, linke, haseloff, zweidrittelmehrheit, richter, sachsen anhalt, anhalt, ministerpräsident, gewählt, brandenburg]"
4,5,24,523,24_russland_ukraine_russischen_russische,"[russland, ukraine, russischen, russische, sanktionen, merz, belgien, euroclear, eu staaten, gas, vermögen, russisches, brüssel, ungarn, kommission, milliarden euro, nutzung, gipfel, für ukraine, russlands, vermögenswerte, leyen, putin, slowakei, wever, de wever, orban, eingefrorenen, lng, bundeskanzler]"
5,6,3,460,3_venezuela_maduro_trump_venezuelas,"[venezuela, maduro, trump, venezuelas, machado, bolsonaro, nicolás, nicolás maduro, karibik, venezolanische, venezolanischen, präsidenten, angriff, caracas, drogen, kolumbien, us präsident, donald trump, öl, donald, us regierung, präsident donald, milei, boote, vereinigten staaten, washington, staatschef, un, brasilien, moraes]"
6,7,40,368,40_wirtschaft_ifo_deutsche wirtschaft_wachstum,"[wirtschaft, ifo, deutsche wirtschaft, wachstum, industrie, investitionen, quartal, ökonomen, deutschen wirtschaft, plus, reformen, wirtschaftsleistung, zölle, ifo institut, bundesregierung, produktion, bruttoinlandsprodukt, rückgang, milliarden euro, bundesamt, bip, konjunktur, für 2026, prognose, statistische, exporte, insolvenzen, statistische bundesamt, reiche, schwache]"
7,8,2,354,2_dollar_dax_fed_anleger,"[dollar, dax, fed, anleger, punkten, aktien, aktie, ki, notenbank, börse, leitindex, quartal, milliarden dollar, nvidia, wall street, wall, street, punkte, handel, konzern, us notenbank, nasdaq, powell, gold, index, marke, dow, zinsen, trump, börsen]"
8,9,28,335,28_merz_kanzler_stadtbild_mercosur,"[merz, kanzler, stadtbild, mercosur, bundeskanzler, afd, friedrich, friedrich merz, abkommen, cdu, grünen, spd, merkel, migration, weber, kommission, kanzler merz, bundesregierung, parlament, sagte merz, leyen, regierungserklärung, bundestag, aussagen, ukraine, bundeskanzler friedrich, kanzlers, französischen, töchter, macron]"
9,10,47,333,47_nato_drohnen_polen_luftraum,"[nato, drohnen, polen, luftraum, polnischen, russland, russische, ukraine, russischen, polens, polnischen luftraum, luftraums, tusk, russische drohnen, vorfall, eindringen, russlands, polnische, belarus, ostflanke, warschau, russischer, kampfjets, drohne, eingedrungen, drohnen polnischen, verletzung, litauen, russischer drohnen, pistorius]"


Merged topic count excluding -1: 64
Decision after inspection: continue, adjust min_similarity, or inspect topic overlaps more closely.


In [11]:
step4_keys = ["tagesschau", "rt", "antispiegel", "tichys", "nius"]
step4_labels = [OUTLET_SPECS[key].label for key in step4_keys]
step4_models = [loaded_models[key] for key in step4_keys]

step4_merged_model = BERTopic.merge_models(step4_models, min_similarity=MIN_SIMILARITY)
step4_topic_info = step4_merged_model.get_topic_info().copy()

step4_substantive = (
    step4_topic_info.loc[step4_topic_info["Topic"] != -1, ["Topic", "Count", "Name", "Representation"]]
    .sort_values(["Count", "Topic"], ascending=[False, True])
    .reset_index(drop=True)
)
step4_substantive.insert(0, "Rank", range(1, len(step4_substantive) + 1))

step4_summary = pd.DataFrame(
    [
        {
            "IncludedOutlets": " + ".join(step4_labels),
            "BaseOutlet": OUTLET_SPECS[step4_keys[0]].label,
            "NewlyAddedOutlet": OUTLET_SPECS[step4_keys[-1]].label,
            "PreviousMergedTopics": int(step3_substantive["Topic"].nunique()),
            "AddedOutletTopics": int(step0_summary.loc[step0_summary["Outlet"] == OUTLET_SPECS["nius"].label, "Topics"].iloc[0]),
            "MergedTopicsNow": int(step4_substantive["Topic"].nunique()),
            "MergedRowsIncludingOutlier": int(len(step4_topic_info)),
        }
    ]
)

display(step4_summary)
display(step4_substantive.head(30))

print("Merged topic count excluding -1:", int(step4_substantive["Topic"].nunique()))
print("Decision after inspection: continue, adjust min_similarity, or inspect topic overlaps more closely.")


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 8046.61it/s]
BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


,IncludedOutlets,BaseOutlet,NewlyAddedOutlet,PreviousMergedTopics,AddedOutletTopics,MergedTopicsNow,MergedRowsIncludingOutlier
0,Tagesschau + RT + Antispiegel + Tichys Einblick + Nius,Tagesschau,Nius,64,38,67,68


,Rank,Topic,Count,Name,Representation
0,1,0,1124,0_israel_hamas_gazastreifen_gaza,"[israel, hamas, gazastreifen, gaza, geiseln, israelischen, israelische, israels, netanjahu, un, armee, waffenruhe, palästinensischen, al, freilassung, angriff, benjamin netanjahu, palästinenser, katar, krieg, benjamin, terrororganisation, getötet, trump, gazastreifens, ägypten, gaza stadt, palästinensische, plan, hisbollah]"
1,2,1,1110,1_ukraine_putin_selenskyj_russland,"[ukraine, putin, selenskyj, russland, trump, russischen, ukrainischen, ukrainische, sicherheitsgarantien, us präsident, russische, präsidenten, nato, moskau, wladimir, wladimir putin, wolodymyr, wolodymyr selenskyj, krieg, donald, donald trump, alaska, frieden, für ukraine, merz, russlands, präsident donald, gespräche, europäer, kiew]"
2,3,15,834,15_bsw_afd_cdu_spd,"[bsw, afd, cdu, spd, grünen, wahl, koalition, bundestag, kandidaten, gersdorf, brosius, brosius gersdorf, wagenknecht, fraktion, kandidatin, schulze, stimmen, sachsen, bundesverfassungsgericht, linken, mehrheit, linke, haseloff, zweidrittelmehrheit, richter, sachsen anhalt, anhalt, ministerpräsident, gewählt, brandenburg]"
3,4,48,711,48_ukraine_russland_russische_russischen,"[ukraine, russland, russische, russischen, ukrainische, ukrainischen, pokrowsk, soldaten, truppen, front, putin, region, selenskyj, krieg, kiew, drohnen, donezk, russischer, armee, belarus, moskau, manöver, russlands, donbass, ukrainer, sapad, telegram, streitkräfte, wolodymyr selenskyj, mariupol]"
4,5,5,616,5_bundesanwaltschaft_gericht_haft_prozess,"[bundesanwaltschaft, gericht, haft, prozess, sarkozy, mann, ermittler, tat, verurteilt, staatsanwaltschaft, urteil, stream, nord stream, ermittlungen, festgenommen, nord, anklage, angeklagten, festnahme, polizei, benko, täter, untersuchungshaft, marsalek, angeklagte, mutmaßlichen, landgericht, krah, anschlag, pipelines]"
5,6,28,554,28_merz_kanzler_stadtbild_mercosur,"[merz, kanzler, stadtbild, mercosur, bundeskanzler, afd, friedrich, friedrich merz, abkommen, cdu, grünen, spd, merkel, migration, weber, kommission, kanzler merz, bundesregierung, parlament, sagte merz, leyen, regierungserklärung, bundestag, aussagen, ukraine, bundeskanzler friedrich, kanzlers, französischen, töchter, macron]"
6,7,24,523,24_russland_ukraine_russischen_russische,"[russland, ukraine, russischen, russische, sanktionen, merz, belgien, euroclear, eu staaten, gas, vermögen, russisches, brüssel, ungarn, kommission, milliarden euro, nutzung, gipfel, für ukraine, russlands, vermögenswerte, leyen, putin, slowakei, wever, de wever, orban, eingefrorenen, lng, bundeskanzler]"
7,8,3,480,3_venezuela_maduro_trump_venezuelas,"[venezuela, maduro, trump, venezuelas, machado, bolsonaro, nicolás, nicolás maduro, karibik, venezolanische, venezolanischen, präsidenten, angriff, caracas, drogen, kolumbien, us präsident, donald trump, öl, donald, us regierung, präsident donald, milei, boote, vereinigten staaten, washington, staatschef, un, brasilien, moraes]"
8,9,40,457,40_wirtschaft_ifo_deutsche wirtschaft_wachstum,"[wirtschaft, ifo, deutsche wirtschaft, wachstum, industrie, investitionen, quartal, ökonomen, deutschen wirtschaft, plus, reformen, wirtschaftsleistung, zölle, ifo institut, bundesregierung, produktion, bruttoinlandsprodukt, rückgang, milliarden euro, bundesamt, bip, konjunktur, für 2026, prognose, statistische, exporte, insolvenzen, statistische bundesamt, reiche, schwache]"
9,10,26,456,26_dobrindt_asylbewerber_grenzkontrollen_geflüchtete,"[dobrindt, asylbewerber, grenzkontrollen, geflüchtete, migration, flüchtlinge, kontrollen, asylverfahren, asylanträge, geflüchteten, migranten, asyl, eugh, innenminister, bundespolizei, unterbringung, einbürgerung, csu, albanien, migrationspolitik, frontex, syrien, abschiebungen, herkunftsländer, asylbewerbern, bundesinnenminister, alexander dobrindt, eu staaten, drittstaaten, bürgergeld]"


Merged topic count excluding -1: 67
Decision after inspection: continue, adjust min_similarity, or inspect topic overlaps more closely.


In [12]:
step5_keys = ["tagesschau", "rt", "antispiegel", "tichys", "nius", "compact"]
step5_labels = [OUTLET_SPECS[key].label for key in step5_keys]
step5_models = [loaded_models[key] for key in step5_keys]

step5_merged_model = BERTopic.merge_models(step5_models, min_similarity=MIN_SIMILARITY)
step5_topic_info = step5_merged_model.get_topic_info().copy()

step5_substantive = (
    step5_topic_info.loc[step5_topic_info["Topic"] != -1, ["Topic", "Count", "Name", "Representation"]]
    .sort_values(["Count", "Topic"], ascending=[False, True])
    .reset_index(drop=True)
)
step5_substantive.insert(0, "Rank", range(1, len(step5_substantive) + 1))

step5_summary = pd.DataFrame(
    [
        {
            "IncludedOutlets": " + ".join(step5_labels),
            "BaseOutlet": OUTLET_SPECS[step5_keys[0]].label,
            "NewlyAddedOutlet": OUTLET_SPECS[step5_keys[-1]].label,
            "PreviousMergedTopics": int(step4_substantive["Topic"].nunique()),
            "AddedOutletTopics": int(step0_summary.loc[step0_summary["Outlet"] == OUTLET_SPECS["compact"].label, "Topics"].iloc[0]),
            "MergedTopicsNow": int(step5_substantive["Topic"].nunique()),
            "MergedRowsIncludingOutlier": int(len(step5_topic_info)),
        }
    ]
)

display(step5_summary)
display(step5_substantive.head(30))

print("Merged topic count excluding -1:", int(step5_substantive["Topic"].nunique()))
print("Decision after inspection: continue, adjust min_similarity, or inspect topic overlaps more closely.")


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 9365.54it/s]
BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


,IncludedOutlets,BaseOutlet,NewlyAddedOutlet,PreviousMergedTopics,AddedOutletTopics,MergedTopicsNow,MergedRowsIncludingOutlier
0,Tagesschau + RT + Antispiegel + Tichys Einblick + Nius + Compact,Tagesschau,Compact,67,31,72,73


,Rank,Topic,Count,Name,Representation
0,1,1,1172,1_ukraine_putin_selenskyj_russland,"[ukraine, putin, selenskyj, russland, trump, russischen, ukrainischen, ukrainische, sicherheitsgarantien, us präsident, russische, präsidenten, nato, moskau, wladimir, wladimir putin, wolodymyr, wolodymyr selenskyj, krieg, donald, donald trump, alaska, frieden, für ukraine, merz, russlands, präsident donald, gespräche, europäer, kiew]"
1,2,0,1157,0_israel_hamas_gazastreifen_gaza,"[israel, hamas, gazastreifen, gaza, geiseln, israelischen, israelische, israels, netanjahu, un, armee, waffenruhe, palästinensischen, al, freilassung, angriff, benjamin netanjahu, palästinenser, katar, krieg, benjamin, terrororganisation, getötet, trump, gazastreifens, ägypten, gaza stadt, palästinensische, plan, hisbollah]"
2,3,15,892,15_bsw_afd_cdu_spd,"[bsw, afd, cdu, spd, grünen, wahl, koalition, bundestag, kandidaten, gersdorf, brosius, brosius gersdorf, wagenknecht, fraktion, kandidatin, schulze, stimmen, sachsen, bundesverfassungsgericht, linken, mehrheit, linke, haseloff, zweidrittelmehrheit, richter, sachsen anhalt, anhalt, ministerpräsident, gewählt, brandenburg]"
3,4,48,778,48_ukraine_russland_russische_russischen,"[ukraine, russland, russische, russischen, ukrainische, ukrainischen, pokrowsk, soldaten, truppen, front, putin, region, selenskyj, krieg, kiew, drohnen, donezk, russischer, armee, belarus, moskau, manöver, russlands, donbass, ukrainer, sapad, telegram, streitkräfte, wolodymyr selenskyj, mariupol]"
4,5,10,705,10_gewalt_frauen_polizei_hubig,"[gewalt, frauen, polizei, hubig, opfer, täter, jugendliche, straftaten, jugendlichen, bka, dobrindt, studie, fälle, ermittlungen, social media, betroffene, media, internet, palantir, kindern, kriminalität, online, sicherheitsbehörden, videos, makler, social, digitale, demokratie, kinder jugendliche, hateaid]"
5,6,5,636,5_bundesanwaltschaft_gericht_haft_prozess,"[bundesanwaltschaft, gericht, haft, prozess, sarkozy, mann, ermittler, tat, verurteilt, staatsanwaltschaft, urteil, stream, nord stream, ermittlungen, festgenommen, nord, anklage, angeklagten, festnahme, polizei, benko, täter, untersuchungshaft, marsalek, angeklagte, mutmaßlichen, landgericht, krah, anschlag, pipelines]"
6,7,28,629,28_merz_kanzler_stadtbild_mercosur,"[merz, kanzler, stadtbild, mercosur, bundeskanzler, afd, friedrich, friedrich merz, abkommen, cdu, grünen, spd, merkel, migration, weber, kommission, kanzler merz, bundesregierung, parlament, sagte merz, leyen, regierungserklärung, bundestag, aussagen, ukraine, bundeskanzler friedrich, kanzlers, französischen, töchter, macron]"
7,8,24,523,24_russland_ukraine_russischen_russische,"[russland, ukraine, russischen, russische, sanktionen, merz, belgien, euroclear, eu staaten, gas, vermögen, russisches, brüssel, ungarn, kommission, milliarden euro, nutzung, gipfel, für ukraine, russlands, vermögenswerte, leyen, putin, slowakei, wever, de wever, orban, eingefrorenen, lng, bundeskanzler]"
8,9,3,516,3_venezuela_maduro_trump_venezuelas,"[venezuela, maduro, trump, venezuelas, machado, bolsonaro, nicolás, nicolás maduro, karibik, venezolanische, venezolanischen, präsidenten, angriff, caracas, drogen, kolumbien, us präsident, donald trump, öl, donald, us regierung, präsident donald, milei, boote, vereinigten staaten, washington, staatschef, un, brasilien, moraes]"
9,10,40,457,40_wirtschaft_ifo_deutsche wirtschaft_wachstum,"[wirtschaft, ifo, deutsche wirtschaft, wachstum, industrie, investitionen, quartal, ökonomen, deutschen wirtschaft, plus, reformen, wirtschaftsleistung, zölle, ifo institut, bundesregierung, produktion, bruttoinlandsprodukt, rückgang, milliarden euro, bundesamt, bip, konjunktur, für 2026, prognose, statistische, exporte, insolvenzen, statistische bundesamt, reiche, schwache]"


Merged topic count excluding -1: 72
Decision after inspection: continue, adjust min_similarity, or inspect topic overlaps more closely.


In [13]:
step6_keys = ["tagesschau", "rt", "antispiegel", "tichys", "nius", "compact", "deutschlandkurier"]
step6_labels = [OUTLET_SPECS[key].label for key in step6_keys]
step6_models = [loaded_models[key] for key in step6_keys]

step6_merged_model = BERTopic.merge_models(step6_models, min_similarity=MIN_SIMILARITY)
step6_topic_info = step6_merged_model.get_topic_info().copy()

step6_substantive = (
    step6_topic_info.loc[step6_topic_info["Topic"] != -1, ["Topic", "Count", "Name", "Representation"]]
    .sort_values(["Count", "Topic"], ascending=[False, True])
    .reset_index(drop=True)
)
step6_substantive.insert(0, "Rank", range(1, len(step6_substantive) + 1))

step6_summary = pd.DataFrame(
    [
        {
            "IncludedOutlets": " + ".join(step6_labels),
            "BaseOutlet": OUTLET_SPECS[step6_keys[0]].label,
            "NewlyAddedOutlet": OUTLET_SPECS[step6_keys[-1]].label,
            "PreviousMergedTopics": int(step5_substantive["Topic"].nunique()),
            "AddedOutletTopics": int(step0_summary.loc[step0_summary["Outlet"] == OUTLET_SPECS["deutschlandkurier"].label, "Topics"].iloc[0]),
            "MergedTopicsNow": int(step6_substantive["Topic"].nunique()),
            "MergedRowsIncludingOutlier": int(len(step6_topic_info)),
        }
    ]
)

display(step6_summary)
display(step6_substantive.head(30))

print("Merged topic count excluding -1:", int(step6_substantive["Topic"].nunique()))
print("Decision after inspection: move to article-level assignment/outlier handling, or revisit merge settings.")


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4855.28it/s]
BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


,IncludedOutlets,BaseOutlet,NewlyAddedOutlet,PreviousMergedTopics,AddedOutletTopics,MergedTopicsNow,MergedRowsIncludingOutlier
0,Tagesschau + RT + Antispiegel + Tichys Einblick + Nius + Compact + Deutschlandkurier,Tagesschau,Deutschlandkurier,72,26,72,73


,Rank,Topic,Count,Name,Representation
0,1,1,1308,1_ukraine_putin_selenskyj_russland,"[ukraine, putin, selenskyj, russland, trump, russischen, ukrainischen, ukrainische, sicherheitsgarantien, us präsident, russische, präsidenten, nato, moskau, wladimir, wladimir putin, wolodymyr, wolodymyr selenskyj, krieg, donald, donald trump, alaska, frieden, für ukraine, merz, russlands, präsident donald, gespräche, europäer, kiew]"
1,2,0,1157,0_israel_hamas_gazastreifen_gaza,"[israel, hamas, gazastreifen, gaza, geiseln, israelischen, israelische, israels, netanjahu, un, armee, waffenruhe, palästinensischen, al, freilassung, angriff, benjamin netanjahu, palästinenser, katar, krieg, benjamin, terrororganisation, getötet, trump, gazastreifens, ägypten, gaza stadt, palästinensische, plan, hisbollah]"
2,3,15,970,15_bsw_afd_cdu_spd,"[bsw, afd, cdu, spd, grünen, wahl, koalition, bundestag, kandidaten, gersdorf, brosius, brosius gersdorf, wagenknecht, fraktion, kandidatin, schulze, stimmen, sachsen, bundesverfassungsgericht, linken, mehrheit, linke, haseloff, zweidrittelmehrheit, richter, sachsen anhalt, anhalt, ministerpräsident, gewählt, brandenburg]"
3,4,10,870,10_gewalt_frauen_polizei_hubig,"[gewalt, frauen, polizei, hubig, opfer, täter, jugendliche, straftaten, jugendlichen, bka, dobrindt, studie, fälle, ermittlungen, social media, betroffene, media, internet, palantir, kindern, kriminalität, online, sicherheitsbehörden, videos, makler, social, digitale, demokratie, kinder jugendliche, hateaid]"
4,5,28,787,28_merz_kanzler_stadtbild_mercosur,"[merz, kanzler, stadtbild, mercosur, bundeskanzler, afd, friedrich, friedrich merz, abkommen, cdu, grünen, spd, merkel, migration, weber, kommission, kanzler merz, bundesregierung, parlament, sagte merz, leyen, regierungserklärung, bundestag, aussagen, ukraine, bundeskanzler friedrich, kanzlers, französischen, töchter, macron]"
5,6,48,778,48_ukraine_russland_russische_russischen,"[ukraine, russland, russische, russischen, ukrainische, ukrainischen, pokrowsk, soldaten, truppen, front, putin, region, selenskyj, krieg, kiew, drohnen, donezk, russischer, armee, belarus, moskau, manöver, russlands, donbass, ukrainer, sapad, telegram, streitkräfte, wolodymyr selenskyj, mariupol]"
6,7,5,749,5_bundesanwaltschaft_gericht_haft_prozess,"[bundesanwaltschaft, gericht, haft, prozess, sarkozy, mann, ermittler, tat, verurteilt, staatsanwaltschaft, urteil, stream, nord stream, ermittlungen, festgenommen, nord, anklage, angeklagten, festnahme, polizei, benko, täter, untersuchungshaft, marsalek, angeklagte, mutmaßlichen, landgericht, krah, anschlag, pipelines]"
7,8,24,564,24_russland_ukraine_russischen_russische,"[russland, ukraine, russischen, russische, sanktionen, merz, belgien, euroclear, eu staaten, gas, vermögen, russisches, brüssel, ungarn, kommission, milliarden euro, nutzung, gipfel, für ukraine, russlands, vermögenswerte, leyen, putin, slowakei, wever, de wever, orban, eingefrorenen, lng, bundeskanzler]"
8,9,26,530,26_dobrindt_asylbewerber_grenzkontrollen_geflüchtete,"[dobrindt, asylbewerber, grenzkontrollen, geflüchtete, migration, flüchtlinge, kontrollen, asylverfahren, asylanträge, geflüchteten, migranten, asyl, eugh, innenminister, bundespolizei, unterbringung, einbürgerung, csu, albanien, migrationspolitik, frontex, syrien, abschiebungen, herkunftsländer, asylbewerbern, bundesinnenminister, alexander dobrindt, eu staaten, drittstaaten, bürgergeld]"
9,10,40,521,40_wirtschaft_ifo_deutsche wirtschaft_wachstum,"[wirtschaft, ifo, deutsche wirtschaft, wachstum, industrie, investitionen, quartal, ökonomen, deutschen wirtschaft, plus, reformen, wirtschaftsleistung, zölle, ifo institut, bundesregierung, produktion, bruttoinlandsprodukt, rückgang, milliarden euro, bundesamt, bip, konjunktur, für 2026, prognose, statistische, exporte, insolvenzen, statistische bundesamt, reiche, schwache]"


Merged topic count excluding -1: 72
Decision after inspection: move to article-level assignment/outlier handling, or revisit merge settings.
